**MNIST VISULIZATION**

In [1]:
!apt-get install -y libpango1.0-dev libcairo2-dev pkg-config python3-dev
!pip install manim numpy
!apt-get install -y texlive texlive-latex-extra texlive-fonts-extra dvipng

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pkg-config is already the newest version (0.29.2-1ubuntu3).
libcairo2-dev is already the newest version (1.16.0-5ubuntu2.1).
libpango1.0-dev is already the newest version (1.50.6+ds-2ubuntu1).
python3-dev is already the newest version (3.10.6-1~22.04.1).
The following packages were automatically installed and are no longer required:
  libbz2-dev libpkgconf3 libreadline-dev
Use 'apt autoremove' to remove them.
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages were automatically installed and are no longer required:
  libbz2-dev libpkgconf3 libreadline-dev
Use 'apt autoremove' to remove them.
The following additional packages will be installed:
  dvisvgm fonts-adf-accanthis fonts-adf-berenis fonts-adf-gillius
  fonts-adf-universalis fonts-cabin fonts-cantarell f

In [2]:
from manim import *
import numpy as np

class DeepLearningMNIST(Scene):
    def construct(self):
        # Set a sleek dark background
        self.camera.background_color = "#0d1117"

        # ==========================================================
        # SCENE 1: THE RAW DATA (28x28 IMAGE)
        # ==========================================================
        title = Text("1. The Input Data", font_size=48, weight=BOLD).to_edge(UP)
        self.play(Write(title))

        # Create a stylized 8x8 grid to represent our 28x28 image
        grid = VGroup(*[
            Square(side_length=0.3, fill_opacity=np.random.uniform(0.1, 0.9), color=TEAL)
            for _ in range(64)
        ]).arrange_in_grid(rows=8, cols=8)

        grid_label = Text("28 x 28 Pixel Grid", font_size=24).next_to(grid, DOWN)

        self.play(FadeIn(grid, shift=UP), Write(grid_label), run_time=2)
        self.wait(1)

        # ==========================================================
        # SCENE 2: FLATTENING THE TENSOR
        # ==========================================================
        title_flat = Text("2. Flattening the Tensor", font_size=48, weight=BOLD).to_edge(UP)
        self.play(Transform(title, title_flat))

        # Morph the 2D grid into a 1D column vector
        flattened_nodes = VGroup(*[
            Circle(radius=0.1, fill_opacity=0.8, color=TEAL, stroke_width=1)
            for _ in range(20) # Proxy for 784
        ]).arrange(DOWN, buff=0.15).to_edge(LEFT, buff=1.5)

        vector_label = Text("784 Input Features", font_size=24).next_to(flattened_nodes, DOWN)

        self.play(
            Transform(grid[:20], flattened_nodes),
            FadeOut(grid[20:]),
            Transform(grid_label, vector_label),
            run_time=2
        )
        self.wait(1)

        # ==========================================================
        # SCENE 3: BUILDING THE ARCHITECTURE
        # ==========================================================
        title_net = Text("3. Deep Neural Network Architecture", font_size=48, weight=BOLD).to_edge(UP)
        self.play(Transform(title, title_net))

        # Layer dimensions (Visual proxy for 784 -> 256 -> 256 -> 10)
        layer_sizes = [20, 12, 12, 10]
        layer_colors = [TEAL, PURPLE, PURPLE, MAROON]
        layer_labels = ["Input", "Hidden 1", "Hidden 2", "Output (0-9)"]

        layers = VGroup()
        for size, color in zip(layer_sizes, layer_colors):
            layer = VGroup(*[Circle(radius=0.1, fill_color=color, fill_opacity=0.8, stroke_width=1) for _ in range(size)])
            layer.arrange(DOWN, buff=0.15)
            layers.add(layer)

        layers.arrange(RIGHT, buff=2).shift(DOWN * 0.2)

        # Snap the flattened inputs into the network
        self.play(Transform(grid[:20], layers[0]), FadeOut(grid_label), run_time=1)

        # Pop in the hidden and output layers
        text_group = VGroup()
        for i in range(1, len(layer_sizes)):
            self.play(FadeIn(layers[i], scale=0.5), run_time=0.5)

        for i, text in enumerate(layer_labels):
            label = Text(text, font_size=20).next_to(layers[i], UP, buff=0.3)
            text_group.add(label)

        self.play(FadeIn(text_group), run_time=1)

        # Draw the dense connections (Synapses)
        edges = VGroup()
        for i in range(len(layers) - 1):
            layer_edges = VGroup()
            for n1 in layers[i]:
                for n2 in layers[i+1]:
                    edge = Line(n1.get_center(), n2.get_center(), stroke_width=0.3, stroke_opacity=0.15, color=WHITE)
                    layer_edges.add(edge)
            edges.add(layer_edges)

        self.play(LaggedStart(*[Create(e) for e in edges], lag_ratio=0.1), run_time=2)
        self.wait(1)

        # ==========================================================
        # SCENE 4: ZOOMING INTO A SINGLE NEURON
        # ==========================================================
        title_zoom = Text("4. Inside the Hidden Node: Weights & Biases", font_size=48, weight=BOLD).to_edge(UP)
        self.play(Transform(title, title_zoom))

        # Focus on one specific node in Hidden Layer 1
        target_node = layers[1][5]

        # Fade out everything else slightly to focus
        self.play(
            layers.animate.set_opacity(0.2),
            edges.animate.set_opacity(0.05),
            text_group.animate.set_opacity(0.2),
            target_node.animate.set_opacity(1).scale(1.5).set_color(YELLOW)
        )

        # Display the Math
        math_z = MathTex(r"z = \sum_{i=1}^{n} (w_i \cdot x_i) + b", font_size=40).to_corner(UR).shift(DOWN * 1.5 + LEFT * 0.5)
        math_a = MathTex(r"a = \sigma(z) = \frac{1}{1 + e^{-z}}", font_size=40).next_to(math_z, DOWN, aligned_edge=LEFT, buff=0.5)

        math_z[0][5:7].set_color(TEAL)   # Weights
        math_z[0][8:10].set_color(BLUE)  # Inputs
        math_z[0][12].set_color(RED)     # Bias

        self.play(Write(math_z))

        # Highlight incoming edges to simulate inputs * weights
        incoming_edges = VGroup(*[Line(n.get_center(), target_node.get_center(), color=TEAL, stroke_width=2) for n in layers[0]])
        self.play(Create(incoming_edges), run_time=1.5)
        self.play(FadeOut(incoming_edges))

        self.play(Write(math_a))
        self.wait(2)

        # Restore the full network view
        self.play(
            FadeOut(math_z), FadeOut(math_a),
            layers.animate.set_opacity(1),
            edges.animate.set_opacity(0.15),
            text_group.animate.set_opacity(1),
            target_node.animate.set_opacity(0.8).scale(1/1.5).set_color(PURPLE)
        )

        # ==========================================================
        # SCENE 5: THE FORWARD PASS (DATA FLOW)
        # ==========================================================
        title_flow = Text("5. Forward Propagation", font_size=48, weight=BOLD).to_edge(UP)
        self.play(Transform(title, title_flow))

        # Send a "pulse" through the network
        self.play(layers[0].animate.set_color(YELLOW), run_time=0.3)
        self.play(edges[0].animate.set_color(YELLOW).set_opacity(0.8), run_time=0.5)
        self.play(layers[0].animate.set_color(TEAL), edges[0].animate.set_color(WHITE).set_opacity(0.15), layers[1].animate.set_color(YELLOW), run_time=0.5)

        self.play(edges[1].animate.set_color(YELLOW).set_opacity(0.8), run_time=0.5)
        self.play(layers[1].animate.set_color(PURPLE), edges[1].animate.set_color(WHITE).set_opacity(0.15), layers[2].animate.set_color(YELLOW), run_time=0.5)

        self.play(edges[2].animate.set_color(YELLOW).set_opacity(0.8), run_time=0.5)
        self.play(layers[2].animate.set_color(PURPLE), edges[2].animate.set_color(WHITE).set_opacity(0.15), layers[3].animate.set_color(GREEN), run_time=0.5)

        self.wait(1)
        self.play(
            FadeOut(layers), FadeOut(edges), FadeOut(text_group), FadeOut(grid[:20])
        )

        # ==========================================================
        # SCENE 6: OPTIMIZATION & TRAINING
        # ==========================================================
        title_train = Text("6. Backpropagation & Training (Adam Optimizer)", font_size=40, weight=BOLD).to_edge(UP)
        self.play(Transform(title, title_train))

        # Dashboard layout
        tracker_epoch = ValueTracker(1)
        tracker_cost = ValueTracker(202.20)
        tracker_acc = ValueTracker(14.6)

        label_epoch = always_redraw(lambda: Text(f"Epoch: {int(tracker_epoch.get_value()):02d} / 25", font_size=48).shift(UP * 0.5))
        label_cost = always_redraw(lambda: Text(f"Cross-Entropy Loss: {tracker_cost.get_value():.2f}", font_size=40, color=RED).next_to(label_epoch, DOWN, buff=0.5))
        label_acc = always_redraw(lambda: Text(f"Test Accuracy: {tracker_acc.get_value():.2f}%", font_size=40, color=GREEN).next_to(label_cost, DOWN, buff=0.5))

        self.play(FadeIn(label_epoch), FadeIn(label_cost), FadeIn(label_acc))

        # Animate the training progress
        self.play(
            tracker_epoch.animate.set_value(25),
            tracker_cost.animate.set_value(76.20),
            tracker_acc.animate.set_value(96.39),
            run_time=5,
            rate_func=linear
        )

        self.wait(1)

        success = Text("Model Successfully Trained!", font_size=56, color=YELLOW, weight=BOLD).shift(DOWN * 2.5)
        self.play(Write(success))
        self.wait(3)

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [4]:
%%manim -qm -v WARNING DeepLearningMNIST

Manim Community v0.20.1

In [9]:
import subprocess
import subprocess
result = subprocess.run(['find', '/', '-name', '*.mp4', '-not', '-path', '*/proc/*'], capture_output=True, text=True)
print(result.stdout)

/usr/share/texlive/texmf-dist/tex/latex/mwe/example-movie.mp4
/usr/local/lib/python3.12/dist-packages/panel/tests/pane/assets/mp4.mp4
/usr/local/lib/python3.12/dist-packages/gradio/media_assets/videos/a.mp4
/usr/local/lib/python3.12/dist-packages/gradio/media_assets/videos/b.mp4
/usr/local/lib/python3.12/dist-packages/gradio/media_assets/videos/world.mp4
/content/media/videos/content/720p30/MNISTVisualization.mp4
/content/media/videos/content/720p30/partial_movie_files/MNISTVisualization/2538612922_3591909666_3692776034.mp4
/content/media/videos/content/720p30/partial_movie_files/MNISTVisualization/2538612922_102892414_4136202213.mp4
/content/media/videos/content/720p30/partial_movie_files/MNISTVisualization/2538612922_1432596116_133749752.mp4
/content/media/videos/content/720p30/partial_movie_files/MNISTVisualization/2538612922_1287027188_528009642.mp4
/content/media/videos/content/720p30/partial_movie_files/MNISTVisualization/2538612922_3981938368_2658725391.mp4
/content/media/videos

In [10]:
from google.colab import files

files.download('/content/media/videos/content/720p30/MNISTVisualization.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>